In [47]:
import pandas as pd
from tqdm import tqdm
import json
from glob import glob
from collections import defaultdict
from pathlib import Path
import re
import json
import pandas as pd

# ==========================================
# 1. SCHEMAS AND MAPPINGS
# ==========================================

# Restored the full ground truth schema
RAW_SCHEMA = {
    "Healthcare": ["Experiencer", "HealthcareType"],
    # "Living": ["Experiencer", "LivingStatus", "LivingType", "ResidentType"],
    "Smoke": ["Experiencer", "SmokeStatus"],
    "Employment": ["EmploymentStatus", "Experiencer"],
    "Social": ["Experiencer", "SocialActivity", "SocialType"],
    "Education": ["EducationStatus", "EducationType", "Experiencer"],
    "Transportation": ["Experiencer", "TransportationConvenienceLevel", "TransportationType"],
    "Mental Health": ["Experiencer", "MentalHealthStatus", "MentalHealthType"],
    "Insurance": ["Experiencer", "InsuranceType"],
    "Financial": ["Experiencer", "FinancialStatus"],
    "Substance Use": ["Experiencer", "SubstanceUseStatus"],
    "Trauma": ["Experiencer", "TraumaStatus", "TraumaType"],
    "Adherence": ["AdherenceLevel", "AdherenceType", "Experiencer"],
    "Literacy": ["Experiencer", "LiteracyLevel", "LiteracyType"],
    "Recommendation": ["RecommendationType"],
    "Concern": ["ConcernLevel"]
}

KEY_MAPPING = {
    "sociallevel": "socialactivity",
    "transportationstatus": "transportationconveniencelevel",
    "educationlevel": "educationtype", 
    "numofcaregivers": "residenttype",
    "recommendationstatus": "recommendationtype"
}

NORMALIZED_SCHEMA = {}
for cat, keys in RAW_SCHEMA.items():
    norm_cat = cat.lower().replace(" ", "").replace("_", "")
    norm_keys = sorted([k.lower().replace(" ", "").replace("_", "") for k in keys])
    NORMALIZED_SCHEMA[norm_cat] = norm_keys

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def clean_for_match(text):
    """
    Strips spaces, punctuation, AND weird Windows encoding artifacts.
    """
    if not text: return ""
    
    # 1. Convert to string
    clean_text = str(text)
    
    # 2. Explicitly destroy the Windows artifact and non-breaking spaces
    clean_text = clean_text.replace('聽', '').replace('\xa0', '')
    
    # 3. Force English/ASCII characters only (strips hidden unicode ghosts)
    clean_text = clean_text.encode('ascii', 'ignore').decode('utf-8')
    
    # 4. Strip punctuation and lowercase
    return re.sub(r'\W+', '', clean_text).lower()

    
def map_experiencer(value):
    val = str(value).lower().strip()
    patient_group = ["patient", "the patient", "patients"]
    caregiver_group = ["father", "mother", "caregiver", "caregivers", 
                       "parents/caregiver", "family", "patients and caregivers", "parents"]
    if val in patient_group: return "patient"
    elif val in caregiver_group: return "caregivers"
    else: return "others"

def normalize_event_to_dict(event_dict):
    normalized_dict = {}
    for key, value in event_dict.items():
        norm_key = key.lower().replace(" ", "").replace("_", "")
        if norm_key == "category": continue
        
        if norm_key in KEY_MAPPING:
            norm_key = KEY_MAPPING[norm_key]
        
        norm_val = str(value).lower().strip().replace("_", " ")
        if norm_key == "experiencer":
            norm_val = map_experiencer(norm_val)
            
        normalized_dict[norm_key] = norm_val
    return normalized_dict




VALUE_MAPPING = {
    "transportation": {
        "transportationconveniencelevel": {"yes": "easy", "no": "hard"},
        "vehicle_access": {"yes": "easy", "no": "hard"},
        "cost": {"low": "easy", "high": "hard"},
        "distance": {"short": "easy", "long": "hard"},
        "license": {"yes": "easy", "no": "hard"},
        "violation": {"no": "easy", "yes": "hard"}
    },
    "financial": {
        "financialstatus": {
            "poverty": "constrain",
            "fundraising": "constrain",
            "pto": "constrain",
            "unpaid time off": "constrain"
        }
    },
    "healthcare": {
        "healthcaretype": {
            "hospital admit": "hospital stay",
            "hospital discharge": "hospital stay",
            "assessment": "clinical visits",
            "transfer": "clinical visits"
        }
    }
}



# ==========================================
# UPDATED NORMALIZER
# ==========================================
def normalize_attribute_values(event_dict, category_name):
    """Normalizes inconsistent prediction values and collapses ground-truth granularity."""
    normalized_dict = {}
    cat_map = VALUE_MAPPING.get(category_name, {})
    
    for key, value in event_dict.items():
        if isinstance(value, str):
            val_lower = value.lower().strip()
            
            # Special dynamic rule for Transportation convenience
            if category_name == "transportation" and key == "transportationconveniencelevel":
                t_type = event_dict.get("transportationtype", "").lower().strip()
                # If it's a yes/no/high/low value, convert it
                if val_lower in ["yes", "no", "low", "high", "short", "long"]:
                    if t_type in ["vehicle access", "license"]:
                        normalized_dict[key] = "easy" if val_lower == "yes" else "hard"
                    elif t_type == "cost":
                        normalized_dict[key] = "easy" if val_lower == "low" else "hard"
                    elif t_type == "distance":
                        normalized_dict[key] = "easy" if val_lower == "short" else "hard"
                    elif t_type == "violation":
                        normalized_dict[key] = "hard" if val_lower == "yes" else "easy"
                else:
                    # PATCH: If it's ALREADY "easy" or "hard", keep it!
                    normalized_dict[key] = val_lower  
                continue # Skip the general mapping below for this specific key
            
            # General mapping rule for Financial, Healthcare, etc.
            if key in cat_map and val_lower in cat_map[key]:
                normalized_dict[key] = cat_map[key][val_lower]
            else:
                normalized_dict[key] = val_lower
        else:
            normalized_dict[key] = value
            
    return normalized_dict

    
# ==========================================
# 3. EVALUATION LOGIC
# ==========================================
def align_and_evaluate(pred_dicts, true_dicts, doc_id, sentence, category, keys_to_evaluate, eval_target_name):
    """
    Generic matching function that only builds tuples from the specified keys.
    """
    tp, fp, fn = 0, 0, 0
    records = []
    
    # Build the tuples based ONLY on the keys we want to evaluate for this pass
    pred_tuples = [tuple([d.get(k, "not_mentioned") for k in keys_to_evaluate]) for d in pred_dicts]
    true_tuples = [tuple([d.get(k, "not_mentioned") for k in keys_to_evaluate]) for d in true_dicts]
    
    matched_true_indices = set()
    matched_pred_indices = set()
    
    # PASS 1: Greedy match
    for i, p_tuple in enumerate(pred_tuples):
        best_match_idx = -1
        best_overlap = 0
        
        for j, t_tuple in enumerate(true_tuples):
            if j in matched_true_indices: continue
            overlap = sum(1 for p_val, t_val in zip(p_tuple, t_tuple) if p_val == t_val)
            if overlap > best_overlap:
                best_overlap = overlap
                best_match_idx = j
                
        if best_match_idx != -1:
            matched_true_indices.add(best_match_idx)
            matched_pred_indices.add(i)
            t_tuple = true_tuples[best_match_idx]
            
            if best_overlap == len(p_tuple):
                tp += 1
                records.append([doc_id, sentence, category, eval_target_name, t_tuple, p_tuple, "Exact Match (TP)"])
            else:
                fp += 1
                fn += 1
                records.append([doc_id, sentence, category, eval_target_name, t_tuple, p_tuple, "Partial Match (FP/FN)"])

    # PASS 2: Complete mismatches
    unmatched_preds = [(i, p) for i, p in enumerate(pred_tuples) if i not in matched_pred_indices]
    unmatched_trues = [(j, t) for j, t in enumerate(true_tuples) if j not in matched_true_indices]
    
    for (i, p_tuple), (j, t_tuple) in zip(unmatched_preds, unmatched_trues):
        matched_pred_indices.add(i)
        matched_true_indices.add(j)
        fp += 1
        fn += 1
        records.append([doc_id, sentence, category, eval_target_name, t_tuple, p_tuple, "Complete Mismatch (FP/FN)"])

    # PASS 3: Hallucinations
    for i, p_tuple in enumerate(pred_tuples):
        if i not in matched_pred_indices:
            fp += 1
            records.append([doc_id, sentence, category, eval_target_name, (), p_tuple, "Hallucination (FP)"])

    # PASS 4: Misses
    for j, t_tuple in enumerate(true_tuples):
        if j not in matched_true_indices:
            fn += 1
            records.append([doc_id, sentence, category, eval_target_name, t_tuple, (), "Missed (FN)"])
            
    return tp, fp, fn, records

def evaluate_single_document(doc_id, true_filepath, pred_filepath):
    with open(true_filepath, 'r', encoding='utf-8') as f: true_data = json.load(f)
    with open(pred_filepath, 'r', encoding='utf-8') as f: pred_data = json.load(f)
        
    doc_exp_metrics = {}
    doc_oth_metrics = {}
    doc_records = []
    
    # --- UPDATE 1: Building the lookup dictionary with clean keys ---
    pred_lookup = {}
    for item in pred_data:
        raw_sentence = item.get("sentence", "")
        match_key = clean_for_match(raw_sentence)  # <-- Using the clean key!
        pred_lookup[match_key] = item.get("extracted_predictions", {})
        
    # --- UPDATE 2: Matching the ground truth with clean keys ---
    for idx, true_item in true_data.items():
        raw_sentence = true_item.get("sentence", "")
        sentence = clean_for_match(raw_sentence)  # <-- Using the same clean key!
        
        true_sdoh_list = true_item.get("SDoH", [])
        
        # Look up the predictions using the clean key!
        pred_extracted = pred_lookup.get(sentence, {}) 
        
        all_categories = set()
        for event in true_sdoh_list: all_categories.update(event.keys())
        all_categories.update(pred_extracted.keys())
        
        for category in all_categories:
            norm_cat = category.lower().replace(" ", "").replace("_", "")
            expected_keys = NORMALIZED_SCHEMA.get(norm_cat, [])
            
            # If the category isn't in our schema, ignore it
            if not expected_keys: continue 
                
            # --- THE REST OF YOUR LOOP STAYS EXACTLY THE SAME BELOW THIS LINE ---
            true_events = [e[category] for e in true_sdoh_list if category in e]
            pred_events = pred_extracted.get(category, {}).get("extracted_conditions", [])
            
            # 1. Normalize structural keys
            pred_dicts = [normalize_event_to_dict(e) for e in pred_events]
            true_dicts = [normalize_event_to_dict(e) for e in true_events]
def evaluate_single_document(doc_id, true_filepath, pred_filepath):
    with open(true_filepath, 'r', encoding='utf-8') as f: true_data = json.load(f,encoding='utf-8')
    with open(pred_filepath, 'r', encoding='utf-8') as f: pred_data = json.load(f,encoding='utf-8')
        
    doc_exp_metrics = {}
    doc_oth_metrics = {}
    doc_records = []
    
    pred_lookup = {}
    for item in pred_data:
        sentence = item.get("sentence", "").strip()
        pred_lookup[sentence] = item.get("extracted_predictions", {})
        
    for idx, true_item in true_data.items():
        sentence = true_item.get("sentence", "").strip()
        true_sdoh_list = true_item.get("SDoH", [])
        pred_extracted = pred_lookup.get(sentence, {})
        
        all_categories = set()
        for event in true_sdoh_list: all_categories.update(event.keys())
        all_categories.update(pred_extracted.keys())
        
        for category in all_categories:
            norm_cat = category.lower().replace(" ", "").replace("_", "")
            expected_keys = NORMALIZED_SCHEMA.get(norm_cat, [])
            
            # If the category isn't in our schema, ignore it
            if not expected_keys: continue 
                
            true_events = [e[category] for e in true_sdoh_list if category in e]
            pred_events = pred_extracted.get(category, {}).get("extracted_conditions", [])


            # # --- START DEBUG SNIPPET ---
            # if true_events and not pred_events:
            #     print(f"WARNING: Missing predictions for category [{category}]")
            #     print(f"Sentence: {sentence[:50]}...")
            #     if not pred_extracted:
            #         print("-> REASON: The sentence was not found in pred_lookup at all!")
            #     else:
            #         print("-> REASON: The sentence was found, but the model generated no 'extracted_conditions'.")
            #     print("-" * 40)
            # # --- END DEBUG SNIPPET ---


            
            
            # 1. Normalize structural keys
            pred_dicts = [normalize_event_to_dict(e) for e in pred_events]
            true_dicts = [normalize_event_to_dict(e) for e in true_events]

            # 2. Normalize the semantic values (APPLY TO BOTH!)
            pred_dicts = [normalize_attribute_values(d, norm_cat) for d in pred_dicts]
            true_dicts = [normalize_attribute_values(d, norm_cat) for d in true_dicts]

            
            # Split keys into Experiencer and Others
            experiencer_keys = [k for k in expected_keys if k == "experiencer"]
            other_keys = [k for k in expected_keys if k != "experiencer"]

            # --- 1. EVALUATE EXPERIENCER ---
            if experiencer_keys:
                tp, fp, fn, records = align_and_evaluate(pred_dicts, true_dicts, doc_id, sentence, category, experiencer_keys, "Experiencer")
                doc_records.extend(records)
                
                if category not in doc_exp_metrics:
                    doc_exp_metrics[category] = {"tp": 0, "fp": 0, "fn": 0}
                doc_exp_metrics[category]["tp"] += tp
                doc_exp_metrics[category]["fp"] += fp
                doc_exp_metrics[category]["fn"] += fn

            # --- 2. EVALUATE OTHER ATTRIBUTES AS A TUPLE ---
            if other_keys:
                tp, fp, fn, records = align_and_evaluate(pred_dicts, true_dicts, doc_id, sentence, category, other_keys, "Other_Attributes")
                doc_records.extend(records)
                
                if category not in doc_oth_metrics:
                    doc_oth_metrics[category] = {"tp": 0, "fp": 0, "fn": 0}
                doc_oth_metrics[category]["tp"] += tp
                doc_oth_metrics[category]["fp"] += fp
                doc_oth_metrics[category]["fn"] += fn
            
    return doc_exp_metrics, doc_oth_metrics, doc_records

def calculate_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def evaluate_entire_corpus(document_file_pairs):
    corpus_exp_metrics = {}
    corpus_oth_metrics = {}
    all_corpus_records = []
    
    for doc_id, true_path, pred_path in document_file_pairs:
        doc_exp, doc_oth, doc_records = evaluate_single_document(doc_id, true_path, pred_path)
        all_corpus_records.extend(doc_records)
        
        # Aggregate Experiencer Metrics
        for category, counts in doc_exp.items():
            if category not in corpus_exp_metrics:
                corpus_exp_metrics[category] = {"tp": 0, "fp": 0, "fn": 0}
            corpus_exp_metrics[category]["tp"] += counts["tp"]
            corpus_exp_metrics[category]["fp"] += counts["fp"]
            corpus_exp_metrics[category]["fn"] += counts["fn"]

        # Aggregate Other Attributes Metrics
        for category, counts in doc_oth.items():
            if category not in corpus_oth_metrics:
                corpus_oth_metrics[category] = {"tp": 0, "fp": 0, "fn": 0}
            corpus_oth_metrics[category]["tp"] += counts["tp"]
            corpus_oth_metrics[category]["fp"] += counts["fp"]
            corpus_oth_metrics[category]["fn"] += counts["fn"]

    def print_metrics_table(title, metrics_dict):
        print(f"=========================================")
        print(f" {title.center(39)} ")
        print(f"=========================================")
        total_tp, total_fp, total_fn = 0, 0, 0
        
        for category, counts in metrics_dict.items():
            tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
            total_tp += tp
            total_fp += fp
            total_fn += fn
            c_p, c_r, c_f1 = calculate_metrics(tp, fp, fn)
            print(f"[{category}] TP: {tp} | FP: {fp} | FN: {fn}  => Precision: {c_p:.4f}, Recall:{c_r:.4f} F1: {c_f1:.4f}")

        print(f"\n--- GLOBAL {title.split()[0].upper()} SCORE ---")
        overall_p, overall_r, overall_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        print(f"Precision: {overall_p:.4f} | Recall: {overall_r:.4f} | F1: {overall_f1:.4f}\n")

    print_metrics_table("EXPERIENCER SCORES", corpus_exp_metrics)
    print_metrics_table("OTHER ATTRIBUTES SCORES", corpus_oth_metrics)

    columns = ["doc_id", "sentence", "category", "target_type", "true_label", "predicted_label", "status"]
    return pd.DataFrame(all_corpus_records, columns=columns)

In [53]:
# ==========================================
# 4. JUPYTER EXECUTION BLOCK
# ==========================================

# Using your exact file parsing logic
output_paths = sorted([Path(p).as_posix() for p in glob("../output/gpt55_type_simple_experiencer/*.json")])
true_paths = sorted([Path(p).as_posix() for p in glob("../data/processed_data/*.json")])

files = []
for i, j in zip(true_paths, output_paths):
    idx = i.rsplit("/", 1)[1].split("_")[0]
    files.append((idx, i, j))

print(f"Discovered {len(files)} document pairs. Starting evaluation...\n")

Discovered 31 document pairs. Starting evaluation...



In [54]:
df_results = evaluate_entire_corpus(files)

            EXPERIENCER SCORES           
[Healthcare] TP: 267 | FP: 107 | FN: 73  => Precision: 0.7139, Recall:0.7853 F1: 0.7479
[Smoke] TP: 22 | FP: 17 | FN: 18  => Precision: 0.5641, Recall:0.5500 F1: 0.5570
[Employment] TP: 142 | FP: 27 | FN: 20  => Precision: 0.8402, Recall:0.8765 F1: 0.8580
[Social] TP: 116 | FP: 54 | FN: 52  => Precision: 0.6824, Recall:0.6905 F1: 0.6864
[Education] TP: 94 | FP: 9 | FN: 7  => Precision: 0.9126, Recall:0.9307 F1: 0.9216
[Transportation] TP: 86 | FP: 53 | FN: 20  => Precision: 0.6187, Recall:0.8113 F1: 0.7020
[Mental Health] TP: 116 | FP: 31 | FN: 18  => Precision: 0.7891, Recall:0.8657 F1: 0.8256
[Insurance] TP: 44 | FP: 14 | FN: 18  => Precision: 0.7586, Recall:0.7097 F1: 0.7333
[Financial] TP: 178 | FP: 39 | FN: 49  => Precision: 0.8203, Recall:0.7841 F1: 0.8018
[Substance Use] TP: 53 | FP: 22 | FN: 23  => Precision: 0.7067, Recall:0.6974 F1: 0.7020
[Trauma] TP: 67 | FP: 51 | FN: 89  => Precision: 0.5678, Recall:0.4295 F1: 0.4891
[Adherence] TP

In [50]:
df_results

,doc_id,sentence,category,target_type,true_label,predicted_label,status
0,102,"102\tPt, *******, is a 4 year old male who is ...",Healthcare,Experiencer,"(patient,)","(patient,)",Exact Match (TP)
1,102,"102\tPt, *******, is a 4 year old male who is ...",Healthcare,Experiencer,"(patient,)",(),Missed (FN)
2,102,"102\tPt, *******, is a 4 year old male who is ...",Healthcare,Other_Attributes,"(diagnosis,)","(clinical visits,)",Complete Mismatch (FP/FN)
3,102,"102\tPt, *******, is a 4 year old male who is ...",Healthcare,Other_Attributes,"(general,)",(),Missed (FN)
4,102,There is no smoking in the home.,Smoke,Experiencer,"(caregivers,)","(patient,)",Complete Mismatch (FP/FN)
...,...,...,...,...,...,...,...
4358,99,Assessment and recommendations: Although t...,Literacy,Other_Attributes,"(high, transplant knowledge)",(),Missed (FN)
4359,99,"However, they state they were advised during t...",Trauma,Experiencer,"(caregivers,)",(),Missed (FN)
4360,99,"However, they state they were advised during t...",Trauma,Other_Attributes,"(past, loss)",(),Missed (FN)
4361,99,Recommendations are listed below: Recommen...,Recommendation,Other_Attributes,"(increase literacy,)","(increase literacy,)",Exact Match (TP)


In [55]:
df_error = df_results[df_results.status != "Exact Match (TP)"]

In [56]:
df_error.to_csv("../output/level_2_error.csv")

In [57]:
df_error.status.value_counts()


status
Complete Mismatch (FP/FN)    588
Hallucination (FP)           444
Missed (FN)                  431
Partial Match (FP/FN)        324
Name: count, dtype: int64

In [9]:
df_error[df_error.category=="Healthcare"]

,doc_id,sentence,category,target_type,true_label,predicted_label,status
1,102,"102\tPt, *******, is a 4 year old male who is ...",Healthcare,Experiencer,"(patient,)",(),Missed (FN)
2,102,"102\tPt, *******, is a 4 year old male who is ...",Healthcare,Other_Attributes,"(diagnosis,)","(clinical visits,)",Complete Mismatch (FP/FN)
3,102,"102\tPt, *******, is a 4 year old male who is ...",Healthcare,Other_Attributes,"(general,)",(),Missed (FN)
30,102,Course of illness: ******* has a history s...,Healthcare,Experiencer,(),"(patient,)",Hallucination (FP)
32,102,Course of illness: ******* has a history s...,Healthcare,Other_Attributes,(),"(surgeries/procedures,)",Hallucination (FP)
...,...,...,...,...,...,...,...
4191,99,Course of illness: ***** was diagnosed wit...,Healthcare,Experiencer,"(patient,)",(),Missed (FN)
4192,99,Course of illness: ***** was diagnosed wit...,Healthcare,Other_Attributes,"(diagnosis,)",(),Missed (FN)
4198,99,Neurology has expressed that this is not a con...,Healthcare,Other_Attributes,"(rehab,)","(clinical visits,)",Complete Mismatch (FP/FN)
4239,99,They also verbalize a commitment to the therap...,Healthcare,Experiencer,"(caregivers,)",(),Missed (FN)
